# Урок 12 - Скорочення історії чату за допомогою Agent Scratchpad

Цей ноутбук демонструє, як керувати контекстом у довгих розмовах за допомогою Microsoft Agent Framework. Зі збільшенням розмови зростає кількість токенів — зрештою це перевищує контекстне вікно моделі. Ми вирішуємо це за допомогою **патерну сумаризації контексту** та **agent scratchpad** для постійної пам’яті.

## Чому ви навчитеся:
1. **Чому важливо керувати контекстом**: розуміння обмежень токенів та контекстних вікон
2. **Агенти, що розуміють контекст**: створення агентів, які керують власним контекстом розмови
3. **Патерн сумаризації контексту**: використання інструментів для скорочення історії розмови
4. **Agent Scratchpad**: постійна пам’ять, яка зберігається при скороченні контексту

## Вимоги:
- Налаштування Azure OpenAI з конфігурованими змінними середовища
- Розуміння базових концепцій агентів з попередніх уроків


## Налаштування


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Чому важливе управління контекстом

Кожен LLM має обмежене **вікно контексту** — максимальну кількість токенів, які він може обробити в одному запиті. В міру розвитку багатокрокової розмови:

- **Кількість токенів зростає лінійно** з кожним повідомленням користувача та відповіддю асистента.
- **Токени підказки домінують у вартості**, оскільки вся історія відправляється знову на кожному кроці.
- Зрештою розмова **перевищує вікно контексту**, і модель або обрізає, або видає помилку.

### Стратегії управління контекстом

| Стратегія | Як це працює | Компроміс |
|---|---|---|
| **Обрізання** | Видаляються найстаріші повідомлення | Втрачається ранній контекст |
| **Підсумовування** | Скорочення старіших повідомлень у резюме | Втрачається частина деталей, але зберігаються ключові моменти |
| **Блокнот / Зовнішня пам’ять** | Збереження ключових фактів поза розмовою | Потрібні виклики інструментів, але інформація зберігається при будь-якому скороченні |

У цій зошиті ми комбінуємо **підсумовування** з інструментом **блокноту**, щоб агент міг підтримувати безперервність навіть при скороченні історії розмови.


## Створення агента, який враховує контекст


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Імітація тривалої розмови

Давайте пройдемо через багатокрокову розмову, щоб побачити, як накопичується контекст. Агент повинен зберігати ключові деталі (переваги, бюджет, дати подорожі) протягом усіх кроків і демонструвати послідовність.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Зауважте, як агент зберігає контекст з попередніх кроків — він знає про Японію, суші, храми, фотографію, бюджет у $3000, подорожі наодинці та період у квітні. У короткій розмові це працює добре, але зі збільшенням тривалості розмови надсилати всю історію стає затратним.

Продовжимо розмову з більшою кількістю кроків, щоб побачити накопичення контексту:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Шаблон узагальнення контексту

У міру розвитку розмови ми можемо використовувати **інструмент узагальнення**, щоб стиснути накопичений контекст у компактний формат. Агент викликає цей інструмент, щоб зафіксувати ключові переваги, щоб навіть якщо старіші повідомлення будуть вилучені, основна інформація збережеться.

Цей шаблон є будівельним блоком для більш складного скорочення історії:
1. Агент визначає ключові факти з розмови
2. Він викликає інструмент узагальнення, щоб зберегти їх
3. Старі повідомлення можна безпечно видаляти, оскільки резюме фіксує найважливіше

Нижче ми визначаємо інструмент `summarize_preferences`, який агент може викликати для запису компактного резюме того, що він дізнався.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Підсумок

У цьому уроці ви навчилися керувати контекстом у тривалих розмовах агентів за допомогою Microsoft Agent Framework:

### Ключові концепції
- **Вікна контексту мають обмеження** — кожен токен у історії розмови коштує грошей і враховується у межі.
- **Інструменти підсумовування** дозволяють агенту стисло викладати накопичений контекст у компактні резюме, зменшуючи використовування токенів при збереженні важливої інформації.
- **Агентські нотатки (scratchpads)** забезпечують постійну зовнішню пам’ять, яка зберігається незалежно від скорочення розмови.

### Що ви створили
- **Агента з урахуванням контексту**, який підтримує безперервність у багатокрокових розмовах
- **Інструмент підсумовування** (`summarize_preferences`), що записує ключові деталі користувача у компактному вигляді
- **Багатокрокову розмову**, що демонструє утримання контексту та обробку змін

### Реальні застосування
- **Боти служби підтримки клієнтів**: запам’ятовують уподобання під час тривалих сесій підтримки
- **Персональні асистенти**: відстежують поточні проєкти без необхідності повторного пояснення контексту
- **Освітні репетитори**: підтримують прогрес студента через багато взаємодій

### Наступні кроки
- Реалізувати повноцінний інструмент нотаток із збереженням у файлах
- Додати автоматичне скорочення історії після підсумовування
- Поєднати з векторними базами даних для семантичного пошуку пам’яті
- Створити агентів, які можуть відновлювати розмови через кілька днів із повним контекстом


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу машинного перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Незважаючи на наші зусилля забезпечити точність, будь ласка, майте на увазі, що автоматичний переклад може містити помилки або неточності. Оригінальний документ його рідною мовою слід вважати авторитетним джерелом. Для отримання критично важливої інформації рекомендується звертатись до професійного людського перекладу. Ми не несемо відповідальності за будь-які непорозуміння чи неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
